In [ ]:

import os, warnings, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import tensorflow as tf
from tensorflow.keras import layers, Model

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


paths_to_try = [
    "/kaggle/input/datasets/salaheddineelkhirani/5-year-data-for-s-and-p-500-and-nasdaq-100/ES_5Years_8_11_2024.csv",
    "/mnt/data/ES_5Years_8_11_2024.csv",
]
csv_path = None
for p in paths_to_try:
    if os.path.exists(p):
        csv_path = p
        break
if csv_path is None:
    for root, _, files in os.walk("/mnt/data"):
        for f in files:
            if f == "ES_5Years_8_11_2024.csv":
                csv_path = os.path.join(root, f)
                break
        if csv_path:
            break
if csv_path is None:
    raise FileNotFoundError("Could not find ES_5Years_8_11_2024.csv in Kaggle or /mnt/data.")

df = pd.read_csv(csv_path)
print("Loaded:", csv_path, "| rows:", len(df), "| cols:", list(df.columns))

# Identify time column + sort (EXACT time-based)

time_col = None
for c in ["Time", "Datetime", "Date", "timestamp"]:
    if c in df.columns:
        time_col = c
        break
if time_col is None:
    raise ValueError("No time column found. Need one of: Time/Datetime/Date/timestamp for exact time-based split.")

df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

need = {"High", "Low"}
if not need.issubset(df.columns):
    raise ValueError(f"Dataset must have {need}. Found: {set(df.columns)}")

has_close = "Close" in df.columns
has_vol = "Volume" in df.columns

print("Time range:", df[time_col].iloc[0], "->", df[time_col].iloc[-1])


#  Time cyclic features (sin/cos)
# - intraday cycle (minute of day / 1440)
# - weekly cycle (day of week / 7)

t = df[time_col]
minute_of_day = (t.dt.hour * 60 + t.dt.minute).astype(int)
dow = t.dt.dayofweek.astype(int)

df["sin_day"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
df["cos_day"] = np.cos(2 * np.pi * minute_of_day / 1440.0)
df["sin_week"] = np.sin(2 * np.pi * dow / 7.0)
df["cos_week"] = np.cos(2 * np.pi * dow / 7.0)

# ---------------------------
# 3) Helpers (labels + features)
# ---------------------------
def slope(y):
    x = np.arange(len(y))
    return np.polyfit(x, y, 1)[0]

def fit_rmse(y):
    x = np.arange(len(y))
    m, b = np.polyfit(x, y, 1)
    yhat = m * x + b
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def range_contraction(highs, lows):
    r = highs - lows
    return -slope(r)

def local_peaks(y, min_sep=4):
    idx = []
    for i in range(1, len(y) - 1):
        if y[i] > y[i - 1] and y[i] > y[i + 1]:
            idx.append(i)
    filt = []
    for i in idx:
        if not filt or (i - filt[-1] >= min_sep):
            filt.append(i)
    return filt

def normalize_window(highs, lows, closes=None):
    mid = (highs + lows) / 2.0
    base = np.median(mid)
    scale = (np.percentile(highs, 95) - np.percentile(lows, 5))
    scale = scale if scale > 1e-9 else 1.0
    H = ((highs - base) / scale).astype(np.float32)
    L = ((lows - base) / scale).astype(np.float32)
    out = {"H": H, "L": L, "base": base, "scale": scale}
    if closes is not None:
        out["C"] = ((closes - base) / scale).astype(np.float32)
    return out

# 3-class labels: 0 Noise, 1 Consolidation/Compression (Converging), 2 DoubleTop
def label_window_3class(highs, lows, level,
                        eps_slope=0.0005, rmse_frac=0.25, dip=0.004, min_conf=1.9):
    """
    FIXED DoubleTop:
    - trough computed from LOW valley (not from highs)
    - dip gate uses level-based threshold for scale consistency (dip * level)
    """
    total_range = float(np.max(highs) - np.min(lows))
    if total_range < level * 0.002:
        return 0, 0.0, 0.0, 0.0

    up_s = slope(highs)
    low_s = slope(lows)
    contr = range_contraction(highs, lows)

    rmse_h = fit_rmse(highs)
    rmse_l = fit_rmse(lows)
    rmse_gate = total_range * rmse_frac
    fit_ok = (rmse_h <= rmse_gate) and (rmse_l <= rmse_gate)

    tol = max(level * 0.0015, 1e-9)

    # Consolidation score
    conv = 0.0
    conv += 1.2 if contr > 0 else -1.2
    if up_s < -eps_slope and low_s > eps_slope:
        conv += 1.3
    if abs(up_s - low_s) > max(abs(up_s), abs(low_s)) * 0.25:
        conv += 0.5
    conv += 0.6 if fit_ok else -0.8

    # DoubleTop score
    dt = 0.0
    pk = local_peaks(highs, min_sep=4)
    if len(pk) >= 2:
        best_pair, best_diff = None, 1e18
        for a in range(len(pk)):
            for b in range(a + 1, len(pk)):
                i1, i2 = pk[a], pk[b]
                if i2 - i1 < 6:
                    continue
                diff = abs(highs[i1] - highs[i2])
                if diff < best_diff:
                    best_diff = diff
                    best_pair = (i1, i2)
        if best_pair is not None:
            i1, i2 = best_pair
            p1, p2 = highs[i1], highs[i2]

            # ✅ FIX: trough from LOW valley
            trough = float(np.min(lows[i1:i2 + 1]))

            # peak similarity
            if abs(p1 - p2) <= tol:
                dt += 1.4

            # ✅ FIX: dip gate uses level-based threshold
            # dip=0.004 => 0.4% of level
            if (min(p1, p2) - trough) >= (dip * level):
                dt += 1.4

            # rebound confirmation (optional)
            if highs[-1] > trough + tol:
                dt += 0.3

    scores = {1: conv, 2: dt}
    best_label = max(scores, key=scores.get)
    best_score = float(scores[best_label])
    y = best_label if best_score >= min_conf else 0
    return y, best_score, conv, dt

def engineered_features(highs, lows, level):
    """
    FIXED DoubleTop geo:
    - trough_depth computed from LOW valley between peaks
    """
    up_s = slope(highs)
    low_s = slope(lows)
    contr = range_contraction(highs, lows)
    rmse_h = fit_rmse(highs) / max(level, 1e-9)
    rmse_l = fit_rmse(lows) / max(level, 1e-9)
    r = highs - lows

    pk = local_peaks(highs, min_sep=4)
    peak_diff = 1.0
    peak_gap = 0.0
    trough_depth = 0.0
    if len(pk) >= 2:
        best_pair, best_diff = None, 1e18
        for a in range(len(pk)):
            for b in range(a + 1, len(pk)):
                i1, i2 = pk[a], pk[b]
                if i2 - i1 < 6:
                    continue
                diff = abs(highs[i1] - highs[i2])
                if diff < best_diff:
                    best_diff = diff
                    best_pair = (i1, i2)
        if best_pair is not None:
            i1, i2 = best_pair
            p1, p2 = highs[i1], highs[i2]

            # ✅ FIX: trough from LOW valley
            trough = float(np.min(lows[i1:i2 + 1]))

            peak_diff = float(abs(p1 - p2) / max(level, 1e-9))
            peak_gap = float((i2 - i1) / len(highs))
            trough_depth = float((min(p1, p2) - trough) / max(level, 1e-9))

    mid = (highs + lows) / 2.0
    ret = np.diff(mid) / np.maximum(mid[:-1], 1e-9)
    vol = float(np.std(ret)) if len(ret) > 0 else 0.0

    return np.array([
        up_s, low_s, contr,
        rmse_h, rmse_l,
        float(np.std(r) / max(level, 1e-9)),
        vol,
        peak_diff, peak_gap, trough_depth
    ], dtype=np.float32)

# 4) BUILD WINDOWS (sequence + geo + labels), keep START TIME for exact split
WINDOW = 30
STRIDE = 30
MIN_CONF = 1.9

X_seq, X_geo, Y, IDX, TSTART = [], [], [], [], []

for start in range(0, len(df) - WINDOW, STRIDE):
    w = df.iloc[start:start + WINDOW]

    highs = w["High"].to_numpy(float)
    lows = w["Low"].to_numpy(float)
    closes = w["Close"].to_numpy(float) if has_close else None
    vols = w["Volume"].to_numpy(float) if has_vol else None

    level = float(np.median((highs + lows) / 2.0))
    if not np.isfinite(level) or level <= 0:
        continue

    y, best_score, conv_s, dt_s = label_window_3class(
        highs, lows, level,
        eps_slope=0.0005, rmse_frac=0.25, dip=0.004, min_conf=MIN_CONF
    )

    normed = normalize_window(highs, lows, closes=closes)

    # base channels
    chans = [normed["H"], normed["L"], (normed["H"] - normed["L"])]

    if has_close:
        chans.append(normed["C"])
        mid = (highs + lows) / 2.0
        ret = np.diff(mid) / np.maximum(mid[:-1], 1e-9)
        ret = np.concatenate([[0.0], ret])
        rscale = np.std(ret) if np.std(ret) > 1e-9 else 1.0
        chans.append((ret / rscale).astype(np.float32))

    # add time encoding channels (sin/cos)
    chans.append(w["sin_day"].to_numpy(np.float32))
    chans.append(w["cos_day"].to_numpy(np.float32))
    chans.append(w["sin_week"].to_numpy(np.float32))
    chans.append(w["cos_week"].to_numpy(np.float32))

    # volume raw (train-only log+scale later)
    if has_vol:
        chans.append(vols.astype(np.float32))

    seq = np.stack(chans, axis=-1).astype(np.float32)
    geo = engineered_features(highs, lows, level)

    X_seq.append(seq)
    X_geo.append(geo)
    Y.append(y)
    IDX.append(start)
    TSTART.append(w[time_col].iloc[0])

X_seq = np.array(X_seq, dtype=np.float32)
X_geo = np.array(X_geo, dtype=np.float32)
Y = np.array(Y, dtype=np.int32)
IDX = np.array(IDX, dtype=np.int32)
TSTART = pd.to_datetime(pd.Series(TSTART)).to_numpy()

print("\nBuilt windows:", len(Y), "| seq shape:", X_seq.shape, "| geo shape:", X_geo.shape)
print("Label distribution:", pd.Series(Y).value_counts().sort_index().to_dict())

# EXACT CALENDAR SPLIT (time-based)

train_end_time = np.datetime64("2023-01-01T00:00:00")
val_end_time   = np.datetime64("2024-01-01T00:00:00")

# sort by start time (chronological)
order = np.argsort(TSTART)
X_seq, X_geo, Y, IDX, TSTART = X_seq[order], X_geo[order], Y[order], IDX[order], TSTART[order]

train_mask = TSTART < train_end_time
val_mask   = (TSTART >= train_end_time) & (TSTART < val_end_time)
test_mask  = TSTART >= val_end_time

train_idx = np.where(train_mask)[0]
val_idx   = np.where(val_mask)[0]
test_idx  = np.where(test_mask)[0]

# Embargo: only needed if overlap exists (STRIDE < WINDOW). Here stride=window => no overlap => embargo=0.
EMBARGO = 0 if STRIDE >= WINDOW else WINDOW
if EMBARGO > 0:
    train_idx = train_idx[train_idx < (train_idx.max() - EMBARGO)]
    val_idx   = val_idx[val_idx < (val_idx.max() - EMBARGO)]

Xseq_tr, Xgeo_tr, y_tr = X_seq[train_idx], X_geo[train_idx], Y[train_idx]
Xseq_va, Xgeo_va, y_va = X_seq[val_idx],   X_geo[val_idx],   Y[val_idx]
Xseq_te, Xgeo_te, y_te = X_seq[test_idx],  X_geo[test_idx],  Y[test_idx]

print("\nSplit sizes:", len(y_tr), len(y_va), len(y_te), "| embargo:", EMBARGO)
print("Train time:", pd.to_datetime(TSTART[train_idx[0]]), "->", pd.to_datetime(TSTART[train_idx[-1]]))
print("Val   time:", pd.to_datetime(TSTART[val_idx[0]]),   "->", pd.to_datetime(TSTART[val_idx[-1]]))
print("Test  time:", pd.to_datetime(TSTART[test_idx[0]]),  "->", pd.to_datetime(TSTART[test_idx[-1]]))

#  PREPROCESS (train-only)
# - geo: RobustScaler
# - volume channel: log1p + RobustScaler

geo_scaler = RobustScaler()
Xgeo_tr_s = geo_scaler.fit_transform(Xgeo_tr)
Xgeo_va_s = geo_scaler.transform(Xgeo_va)
Xgeo_te_s = geo_scaler.transform(Xgeo_te)

def log1p_safe(a):
    return np.log1p(np.maximum(a, 0.0))

if has_vol:
    vol_ch = Xseq_tr.shape[-1] - 1
    v_tr = log1p_safe(Xseq_tr[..., vol_ch]).reshape(-1, 1)
    v_va = log1p_safe(Xseq_va[..., vol_ch]).reshape(-1, 1)
    v_te = log1p_safe(Xseq_te[..., vol_ch]).reshape(-1, 1)

    vol_scaler = RobustScaler()
    v_tr_s = vol_scaler.fit_transform(v_tr).reshape(Xseq_tr.shape[0], WINDOW)
    v_va_s = vol_scaler.transform(v_va).reshape(Xseq_va.shape[0], WINDOW)
    v_te_s = vol_scaler.transform(v_te).reshape(Xseq_te.shape[0], WINDOW)

    Xseq_tr_s = Xseq_tr.copy()
    Xseq_va_s = Xseq_va.copy()
    Xseq_te_s = Xseq_te.copy()
    Xseq_tr_s[..., vol_ch] = v_tr_s
    Xseq_va_s[..., vol_ch] = v_va_s
    Xseq_te_s[..., vol_ch] = v_te_s
else:
    Xseq_tr_s, Xseq_va_s, Xseq_te_s = Xseq_tr, Xseq_va, Xseq_te

NUM_CLASSES = 3
y_tr_oh = tf.keras.utils.to_categorical(y_tr, NUM_CLASSES)
y_va_oh = tf.keras.utils.to_categorical(y_va, NUM_CLASSES)
y_te_oh = tf.keras.utils.to_categorical(y_te, NUM_CLASSES)

# lighter class weights (avoid extreme DT swings)
counts = np.bincount(y_tr, minlength=NUM_CLASSES).astype(float)
cw = {0: 1.0, 1: 1.5, 2: 2.5}
print("\nTrain class counts:", counts, "| class weights:", cw)

#  ADVANCED HYBRID v2 (CNN + Dilated + BN + Transformer(1) + Geo MLP), AdamW, label smoothing

def transformer_encoder(x, d_model, num_heads, d_ff, dropout=0.1):
    attn = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout
    )(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)
    ff = layers.Dense(d_ff, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization()(x)
    return x

seq_in = layers.Input(shape=(WINDOW, Xseq_tr_s.shape[-1]), name="seq_in")
geo_in = layers.Input(shape=(Xgeo_tr_s.shape[-1],), name="geo_in")

x = layers.Conv1D(64, 3, padding="same", activation="relu")(seq_in)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.10)(x)

x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.10)(x)

# dilated conv to enlarge receptive field without too many params
x = layers.Conv1D(64, 3, padding="same", dilation_rate=2, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.10)(x)

# project to d_model then transformer (1 block)
d_model = 64
x = layers.Dense(d_model)(x)
x = transformer_encoder(x, d_model=d_model, num_heads=4, d_ff=128, dropout=0.10)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.10)(x)

# geo branch
g = layers.Dense(64, activation="relu")(geo_in)
g = layers.Dropout(0.20)(g)
g = layers.Dense(32, activation="relu")(g)

# fusion
z = layers.Concatenate()([x, g])
z = layers.Dense(64, activation="relu")(z)
z = layers.Dropout(0.25)(z)
out = layers.Dense(NUM_CLASSES, activation="softmax")(z)

model = Model([seq_in, geo_in], out)

# AdamW fallback-safe
try:
    opt = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)
except Exception:
    try:
        opt = tf.keras.optimizers.experimental.AdamW(learning_rate=1e-3, weight_decay=1e-4)
    except Exception:
        opt = tf.keras.optimizers.Adam(learning_rate=1e-3)

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
model.compile(optimizer=opt, loss=loss_fn, metrics=["accuracy"])
model.summary()

#  Macro-F1 callback + EarlyStopping/ReduceLR on val_macro_f1

def macro_f1_np(y_true_oh, y_pred_prob, num_classes=3):
    y_true = np.argmax(y_true_oh, axis=1)
    y_pred = np.argmax(y_pred_prob, axis=1)
    f1s = []
    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        prec = tp / (tp + fp + 1e-9)
        rec = tp / (tp + fn + 1e-9)
        f1 = 2 * prec * rec / (prec + rec + 1e-9)
        f1s.append(f1)
    return float(np.mean(f1s))

class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.val_data = val_data
        self.best = -1.0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        (xs, xg), yv = self.val_data
        pred = self.model.predict([xs, xg], verbose=0)
        f1 = macro_f1_np(yv, pred, NUM_CLASSES)
        logs["val_macro_f1"] = f1
        print(f" — val_macro_f1: {f1:.4f}")
        if f1 > self.best:
            self.best = f1
            self.best_weights = self.model.get_weights()

mf1_cb = MacroF1Callback(val_data=([Xseq_va_s, Xgeo_va_s], y_va_oh))

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_macro_f1", mode="max", patience=10, restore_best_weights=False
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_macro_f1", mode="max", factor=0.5, patience=4, min_lr=1e-5, verbose=1
)

hist = model.fit(
    [Xseq_tr_s, Xgeo_tr_s], y_tr_oh,
    validation_data=([Xseq_va_s, Xgeo_va_s], y_va_oh),
    epochs=60,
    batch_size=64,
    class_weight=cw,
    callbacks=[mf1_cb, early, reduce_lr],
    verbose=1
)

if mf1_cb.best_weights is not None:
    model.set_weights(mf1_cb.best_weights)
print("\nBest val macro-F1:", mf1_cb.best)

# 10) TEST EVAL

proba = model.predict([Xseq_te_s, Xgeo_te_s], verbose=0)
y_pred = np.argmax(proba, axis=1)

print("\nClassification report (TEST):")
print(classification_report(y_te, y_pred, digits=4, target_names=["Noise", "Consolidation", "DoubleTop"]))

cm = confusion_matrix(y_te, y_pred)
print("Confusion matrix:\n", cm)

try:
    auc = roc_auc_score(y_te_oh, proba, multi_class="ovr")
    print("OVR ROC-AUC:", float(auc))
except Exception as e:
    print("AUC not computed:", e)

plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title("")
plt.xticks([0, 1, 2], ["Noise", "Consol", "DT"])
plt.yticks([0, 1, 2], ["Noise", "Consol", "DT"])
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()

# Timeline probability plot 
plt.figure(figsize=(12, 4))
plt.plot(proba[:, 0], label="P(Noise)")
plt.plot(proba[:, 1], label="P(Consolidation)")
plt.plot(proba[:, 2], label="P(DoubleTop)")
plt.title("Test timeline: class probabilities per window (chronological)")
plt.xlabel("Test window index")
plt.ylabel("Probability")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:


import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    matthews_corrcoef
)

class_names = ["Noise", "Consolidation", "DoubleTop"]

# One-hot true labels
y_true_oh = np.eye(3)[y_te]

# ----------------------------------------------------------
# Average Precision
# ----------------------------------------------------------
ap_scores = {}

for c, name in enumerate(class_names):
    ap_scores[name] = average_precision_score(
        y_true_oh[:, c],
        proba[:, c]
    )

macro_ap = average_precision_score(
    y_true_oh,
    proba,
    average="macro"
)

micro_ap = average_precision_score(
    y_true_oh,
    proba,
    average="micro"
)

# MCC from the SAME predictions
mcc = matthews_corrcoef(y_te, y_pred)

print("\n===== PRECISION-RECALL METRICS =====")
print(f"Noise AP         : {ap_scores['Noise']:.4f}")
print(f"Consolidation AP : {ap_scores['Consolidation']:.4f}")
print(f"DoubleTop AP     : {ap_scores['DoubleTop']:.4f}")
print(f"Macro AP         : {macro_ap:.4f}")
print(f"Micro AP         : {micro_ap:.4f}")
print(f"MCC              : {mcc:.4f}")

# ----------------------------------------------------------
# Figure
# ----------------------------------------------------------

colors = {
    "Noise": "#174A8B",
    "Consolidation": "#008C82",
    "DoubleTop": "#D1495B"
}

micro_color = "#6C4AB6"

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Class-wise PR curves
for c, name in enumerate(class_names):

    precision, recall, _ = precision_recall_curve(
        y_true_oh[:, c],
        proba[:, c]
    )

    ax.plot(
        recall,
        precision,
        linewidth=2.8,
        color=colors[name],
        label=f"{name} (AP = {ap_scores[name]:.4f})"
    )

# Micro-average PR
precision_micro, recall_micro, _ = precision_recall_curve(
    y_true_oh.ravel(),
    proba.ravel()
)

ax.plot(
    recall_micro,
    precision_micro,
    linestyle="--",
    linewidth=2.5,
    color=micro_color,
    label=f"Micro-average (AP = {micro_ap:.4f})"
)

# ----------------------------------------------------------
# Formatting
# ----------------------------------------------------------

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

ax.set_xlabel("Recall", fontsize=12, fontweight="bold")
ax.set_ylabel("Precision", fontsize=12, fontweight="bold")

ax.set_title(
    "Precision-Recall Curves of PatternNet",
    fontsize=13,
    fontweight="bold",
    pad=12
)

ax.grid(
    True,
    linestyle="--",
    linewidth=0.7,
    alpha=0.20
)

# Combined legend
legend = ax.legend(
    loc="lower left",
    fontsize=9.5,
    frameon=True,
    framealpha=1.0,
    fancybox=True,
    borderpad=0.8
)

legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("#C9CDD2")

# ----------------------------------------------------------
# Summary box: Macro AP + MCC
# ----------------------------------------------------------

summary_text = (
    f"Macro AP = {macro_ap:.4f}\n"
    f"MCC = {mcc:.4f}"
)

ax.text(
    0.98,
    0.05,
    summary_text,
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="white",
        edgecolor="#C9CDD2"
    )
)

plt.tight_layout()

# PNG
plt.savefig(
    "Figure7_PatternNet_PR.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

# PDF
plt.savefig(
    "Figure7_PatternNet_PR.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("\nSaved successfully:")
print("Figure7_PatternNet_PR.png")
print("Figure7_PatternNet_PR.pdf")

In [ ]:


import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import shap

# ---------- output paths ----------
OUT_ROOT = "paper_outputs"
OUT_DIR  = os.path.join(OUT_ROOT, "results", "figures")
os.makedirs(OUT_DIR, exist_ok=True)

FIG_PNG_NAME = "shap_geo_summary_polished.png"
FIG_PDF_NAME = "shap_geo_summary_polished.pdf"

FIG_PNG_PATH = os.path.join(OUT_DIR, FIG_PNG_NAME)
FIG_PDF_PATH = os.path.join(OUT_DIR, FIG_PDF_NAME)

ZIP_NAME = "paper_outputs_shap_polished.zip"
ZIP_PATH = os.path.join(OUT_ROOT, ZIP_NAME)

# ---------- class + feature names ----------
CLASS_NAMES = ["Noise", "Consolidation", "DoubleTop"]

# exact engineered feature order from your main code
GEO_FEATURE_NAMES = [
    "slope_high",
    "slope_low",
    "range_contraction",
    "rmse_high_level",
    "rmse_low_level",
    "range_std_level",
    "ret_vol",
    "peak_diff_level",
    "peak_gap_norm",
    "trough_depth_level"
]

# prettier display labels for the paper figure
FEATURE_DISPLAY_NAMES = [
    "Upper slope",
    "Lower slope",
    "Range contraction",
    "Upper-envelope RMSE",
    "Lower-envelope RMSE",
    "Range variability",
    "Return volatility",
    "Peak difference",
    "Peak gap",
    "Trough depth"
]

assert Xgeo_te_s.shape[1] == len(GEO_FEATURE_NAMES), "GEO feature names mismatch"

# ---------- geo-only predictor ----------
WINDOW = int(WINDOW)
SEQ_CH = int(Xseq_te_s.shape[-1])

def predict_proba_from_geo(Xgeo):
    Xgeo = np.array(Xgeo, dtype=np.float32)
    Xseq_zero = np.zeros((Xgeo.shape[0], WINDOW, SEQ_CH), dtype=np.float32)
    return model.predict([Xseq_zero, Xgeo], verbose=0)

# ---------- choose background + explain sets ----------
rng = np.random.default_rng(42)

bg_n = min(200, len(Xgeo_tr_s))
ex_n = min(600, len(Xgeo_te_s))

bg_idx = rng.choice(len(Xgeo_tr_s), size=bg_n, replace=False)
ex_idx = rng.choice(len(Xgeo_te_s), size=ex_n, replace=False)

X_bg = Xgeo_tr_s[bg_idx]
X_ex = Xgeo_te_s[ex_idx]

# ---------- SHAP explainer ----------
explainer = shap.KernelExplainer(predict_proba_from_geo, X_bg)
shap_values_raw = explainer.shap_values(X_ex, nsamples=200)

# ---------- normalize SHAP output format ----------
def shap_to_class_arrays(shap_values, n_classes):
    """
    Returns list of arrays: [class0_array, class1_array, ...]
    each shape = (n_samples, n_features)
    """
    if isinstance(shap_values, list):
        return [np.asarray(v) for v in shap_values]

    arr = np.asarray(shap_values)

    # possible shapes:
    # (n_classes, n_samples, n_features)
    # (n_samples, n_features, n_classes)
    if arr.ndim == 3 and arr.shape[0] == n_classes:
        return [arr[i] for i in range(n_classes)]
    elif arr.ndim == 3 and arr.shape[-1] == n_classes:
        return [arr[:, :, i] for i in range(n_classes)]
    else:
        raise ValueError(f"Unexpected SHAP output shape: {arr.shape}")

class_shap = shap_to_class_arrays(shap_values_raw, len(CLASS_NAMES))

# ---------- choose one SHAP vector per sample ----------
# Use SHAP values for the predicted class from the same geo-only predictor
proba_ex = predict_proba_from_geo(X_ex)
pred_class = np.argmax(proba_ex, axis=1)

selected_shap = np.vstack([
    class_shap[pred_class[i]][i] for i in range(len(X_ex))
])  # shape: (n_samples, n_features)

# ---------- aggregate importance ----------
mean_abs_shap = np.mean(np.abs(selected_shap), axis=0)
sort_idx = np.argsort(mean_abs_shap)[::-1]

feat_names_sorted = [FEATURE_DISPLAY_NAMES[i] for i in sort_idx]
shap_sorted = selected_shap[:, sort_idx]
xval_sorted = X_ex[:, sort_idx]
importance_sorted = mean_abs_shap[sort_idx]

# ---------- custom color map (similar rich academic palette) ----------
feature_cmap = LinearSegmentedColormap.from_list(
    "feature_value_rich",
    ["#1c4e80", "#1fa187", "#d9b44a", "#d95d39"]
)

# ---------- helper for beeswarm-style jitter ----------
def jitter_offsets(n, width=0.18, seed=42):
    local_rng = np.random.default_rng(seed)
    return local_rng.uniform(-width, width, size=n)

# ---------- figure ----------
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.labelweight": "bold",
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

fig = plt.figure(figsize=(13.5, 7.2), dpi=240)
gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.85], wspace=0.08)

ax_bar = fig.add_subplot(gs[0, 0])
ax_swarm = fig.add_subplot(gs[0, 1])

# ---------- left: global importance bar ----------
y_pos = np.arange(len(feat_names_sorted))

bar_colors = ["#2f7f95"] * len(feat_names_sorted)
for i in range(min(3, len(bar_colors))):
    bar_colors[i] = "#2b7a8f"

ax_bar.barh(y_pos, importance_sorted, height=0.62, color=bar_colors, edgecolor="none")
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(feat_names_sorted)
ax_bar.invert_yaxis()
ax_bar.set_xlabel("Mean |SHAP value|")
ax_bar.set_title("Global importance")
ax_bar.grid(axis="x", linestyle="--", alpha=0.18)
ax_bar.set_axisbelow(True)

# clean spines
for spine in ["top", "right", "left"]:
    ax_bar.spines[spine].set_visible(False)
ax_bar.spines["bottom"].set_alpha(0.35)

# numeric labels at bar ends
x_pad = max(importance_sorted) * 0.02
for y, val in zip(y_pos, importance_sorted):
    ax_bar.text(val + x_pad, y, f"{val:.3f}", va="center", ha="left", fontsize=9, color="#324b5c")

# ---------- right: direction and distribution ----------
ax_swarm.set_title("Direction and distribution")
ax_swarm.axvline(0, color="#708090", linestyle="--", linewidth=1.1, alpha=0.75)
ax_swarm.grid(axis="x", linestyle="--", alpha=0.15)
ax_swarm.set_axisbelow(True)

for spine in ["top", "right", "left"]:
    ax_swarm.spines[spine].set_visible(False)
ax_swarm.spines["bottom"].set_alpha(0.35)

# plot one row per feature
for row, feat_idx in enumerate(range(len(feat_names_sorted))):
    shap_vals = shap_sorted[:, feat_idx]
    feat_vals = xval_sorted[:, feat_idx]

    # normalize feature values row-wise for color mapping
    fmin, fmax = np.min(feat_vals), np.max(feat_vals)
    if np.isclose(fmax, fmin):
        norm_vals = np.full_like(feat_vals, 0.5, dtype=float)
    else:
        norm_vals = (feat_vals - fmin) / (fmax - fmin)

    # slight y jitter
    y = np.full_like(shap_vals, row, dtype=float) + jitter_offsets(len(shap_vals), width=0.17, seed=42 + row)

    # sort by feature value so colors spread nicely
    order = np.argsort(norm_vals)
    ax_swarm.scatter(
        shap_vals[order],
        y[order],
        c=norm_vals[order],
        cmap=feature_cmap,
        s=18,
        alpha=0.92,
        edgecolors="none",
        rasterized=True
    )

ax_swarm.set_yticks(y_pos)
ax_swarm.set_yticklabels([])
ax_swarm.set_xlabel("SHAP contribution to predicted class")
ax_swarm.set_ylim(len(feat_names_sorted) - 0.35, -0.65)

# better x-limits with padding
xmin = np.min(shap_sorted)
xmax = np.max(shap_sorted)
xpad = max((xmax - xmin) * 0.06, 0.05)
ax_swarm.set_xlim(xmin - xpad, xmax + xpad)

# ---------- color bar ----------
sm = ScalarMappable(norm=Normalize(vmin=0, vmax=1), cmap=feature_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_swarm, fraction=0.035, pad=0.02)
cbar.set_label("Feature value", rotation=90, labelpad=10, fontweight="bold")
cbar.set_ticks([0, 1])
cbar.set_ticklabels(["Low", "High"])

# ---------- overall spacing ----------
plt.tight_layout()

# ---------- save files ----------
fig.savefig(FIG_PNG_PATH, bbox_inches="tight", facecolor="white")
fig.savefig(FIG_PDF_PATH, bbox_inches="tight", facecolor="white")
plt.show()


print("Saved PNG:", FIG_PNG_PATH)
print("Saved PDF:", FIG_PDF_PATH)


# ---------- ZIP both files ----------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zf.write(FIG_PNG_PATH, arcname=os.path.join("results", "figures", FIG_PNG_NAME))
    zf.write(FIG_PDF_PATH, arcname=os.path.join("results", "figures", FIG_PDF_NAME))

print("Saved ZIP:", ZIP_PATH)


In [ ]:


import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


SEEDS = [123, 777, 2024, 999]


def build_patternnet():

    def transformer_encoder(x, d_model, num_heads, d_ff, dropout=0.1):

        attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout
        )(x, x)

        x = layers.Add()([x, attn])
        x = layers.LayerNormalization()(x)

        ff = layers.Dense(d_ff, activation="relu")(x)
        ff = layers.Dropout(dropout)(ff)
        ff = layers.Dense(d_model)(ff)

        x = layers.Add()([x, ff])
        x = layers.LayerNormalization()(x)

        return x

    seq_in = layers.Input(
        shape=(WINDOW, Xseq_tr_s.shape[-1]),
        name="seq_in"
    )

    geo_in = layers.Input(
        shape=(Xgeo_tr_s.shape[-1],),
        name="geo_in"
    )

    # Sequence branch
    x = layers.Conv1D(
        64, 3, padding="same", activation="relu"
    )(seq_in)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.10)(x)

    x = layers.Conv1D(
        64, 5, padding="same", activation="relu"
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.10)(x)

    x = layers.Conv1D(
        64, 3,
        padding="same",
        dilation_rate=2,
        activation="relu"
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.10)(x)

    d_model = 64

    x = layers.Dense(d_model)(x)

    x = transformer_encoder(
        x,
        d_model=64,
        num_heads=4,
        d_ff=128,
        dropout=0.10
    )

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.10)(x)

    # Geometric branch
    g = layers.Dense(64, activation="relu")(geo_in)
    g = layers.Dropout(0.20)(g)
    g = layers.Dense(32, activation="relu")(g)

    # Fusion
    z = layers.Concatenate()([x, g])
    z = layers.Dense(64, activation="relu")(z)
    z = layers.Dropout(0.25)(z)

    out = layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )(z)

    m = Model([seq_in, geo_in], out)

    try:
        opt = tf.keras.optimizers.AdamW(
            learning_rate=1e-3,
            weight_decay=1e-4
        )
    except Exception:
        opt = tf.keras.optimizers.Adam(
            learning_rate=1e-3
        )

    loss_fn = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.05
    )

    m.compile(
        optimizer=opt,
        loss=loss_fn,
        metrics=["accuracy"]
    )

    return m


# ------------------------------------------------------------
# Multi-seed training
# ------------------------------------------------------------
seed_results = []

for seed in SEEDS:

    print("\n" + "=" * 65)
    print("RUNNING SEED:", seed)
    print("=" * 65)

    tf.keras.backend.clear_session()

    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    model_seed = build_patternnet()

    mf1_seed = MacroF1Callback(
        val_data=(
            [Xseq_va_s, Xgeo_va_s],
            y_va_oh
        )
    )

    early_seed = tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=10,
        restore_best_weights=False
    )

    reduce_seed = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_macro_f1",
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
        verbose=0
    )

    model_seed.fit(
        [Xseq_tr_s, Xgeo_tr_s],
        y_tr_oh,
        validation_data=(
            [Xseq_va_s, Xgeo_va_s],
            y_va_oh
        ),
        epochs=60,
        batch_size=64,
        class_weight=cw,
        callbacks=[
            mf1_seed,
            early_seed,
            reduce_seed
        ],
        verbose=0
    )

    # Restore best validation Macro-F1 weights
    if mf1_seed.best_weights is not None:
        model_seed.set_weights(
            mf1_seed.best_weights
        )

    # Test predictions
    prob_seed = model_seed.predict(
        [Xseq_te_s, Xgeo_te_s],
        verbose=0
    )

    pred_seed = np.argmax(
        prob_seed,
        axis=1
    )

    acc = accuracy_score(
        y_te,
        pred_seed
    )

    precision = precision_score(
        y_te,
        pred_seed,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_te,
        pred_seed,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_te,
        pred_seed,
        average="macro",
        zero_division=0
    )

    dt_f1 = f1_score(
        (y_te == 2).astype(int),
        (pred_seed == 2).astype(int),
        zero_division=0
    )

    auc = roc_auc_score(
        y_te_oh,
        prob_seed,
        multi_class="ovr",
        average="macro"
    )

    seed_results.append({
        "Seed": seed,
        "Accuracy": acc,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": macro_f1,
        "DoubleTop F1": dt_f1,
        "Macro ROC-AUC": auc
    })

    print(
        f"Accuracy={acc:.4f} | "
        f"Macro F1={macro_f1:.4f} | "
        f"DoubleTop F1={dt_f1:.4f} | "
        f"AUC={auc:.4f}"
    )


# ------------------------------------------------------------
# Individual seed results
# ------------------------------------------------------------
seed_df = pd.DataFrame(seed_results)

print("\n")
print("=" * 75)
print("INDIVIDUAL SEED RESULTS")
print("=" * 75)

print(
    seed_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# Mean ± standard deviation
# ------------------------------------------------------------
metrics = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "DoubleTop F1",
    "Macro ROC-AUC"
]

summary_rows = []

for metric in metrics:

    mean_value = seed_df[metric].mean()
    std_value = seed_df[metric].std(ddof=1)

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Std": std_value,
        "Mean ± SD":
            f"{mean_value:.4f} ± {std_value:.4f}"
    })

seed_summary = pd.DataFrame(summary_rows)

print("\n")
print("=" * 75)
print("MULTI-SEED STABILITY: MEAN ± STANDARD DEVIATION")
print("=" * 75)

print(seed_summary.to_string(index=False))


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
seed_df.to_csv(
    "PatternNet_MultiSeed_Individual.csv",
    index=False
)

seed_summary.to_csv(
    "PatternNet_MultiSeed_Summary.csv",
    index=False
)

print("\nSaved:")
print("PatternNet_MultiSeed_Individual.csv")
print("PatternNet_MultiSeed_Summary.csv")

In [ ]:


import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model

# -----------------------------
# 1. Settings
# -----------------------------
TSNE_SEED = 42
PERPLEXITY = 35
N_ITER = 1500

# Class labels
class_names = {
    0: "Noise",
    1: "Consolidation",
    2: "DoubleTop"
}

# Demo-like color palette
class_colors = {
    0: "#27B0A8",   # teal
    1: "#E7BE4A",   # warm gold
    2: "#EF7D57"    # coral / orange
}

# -----------------------------
# 2. Helper: extract embeddings
# -----------------------------
def get_embedding_model(trained_model):
    """
    Uses the penultimate layer output as latent representation.
    """
    return Model(
        inputs=trained_model.inputs,
        outputs=trained_model.layers[-2].output
    )

# Full model latent vectors
embed_model_full = get_embedding_model(model)
Z_full = embed_model_full.predict([Xseq_te_s, Xgeo_te_s], verbose=0)

# Reduced-geometry model latent vectors
embed_model_reduced = get_embedding_model(model_ctrl)
Z_reduced = embed_model_reduced.predict([Xseq_te_s, Xgeo_te_ctrl_s], verbose=0)

print("Full embedding shape   :", Z_full.shape)
print("Reduced embedding shape:", Z_reduced.shape)

# -----------------------------
# 3. Standardize embeddings
# -----------------------------
Z_full_std = StandardScaler().fit_transform(Z_full)
Z_reduced_std = StandardScaler().fit_transform(Z_reduced)

# -----------------------------
# 4. t-SNE projections
# -----------------------------
tsne_full = TSNE(
    n_components=2,
    perplexity=PERPLEXITY,
    n_iter=N_ITER,
    init="pca",
    learning_rate="auto",
    random_state=TSNE_SEED
)

tsne_reduced = TSNE(
    n_components=2,
    perplexity=PERPLEXITY,
    n_iter=N_ITER,
    init="pca",
    learning_rate="auto",
    random_state=TSNE_SEED
)

Y_full = tsne_full.fit_transform(Z_full_std)
Y_reduced = tsne_reduced.fit_transform(Z_reduced_std)

# -----------------------------
# 5. Plot
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 7), dpi=180)

titles = [
    "A   Learned representation of the full PatternNet",
    "B   Learned representation after reduced geometric input"
]

data_list = [Y_full, Y_reduced]

for ax, Y, ttl in zip(axes, data_list, titles):

    for cls in [0, 1, 2]:
        idx = (y_te == cls)
        ax.scatter(
            Y[idx, 0],
            Y[idx, 1],
            s=46,
            c=class_colors[cls],
            label=class_names[cls],
            alpha=0.82,
            edgecolors="white",
            linewidths=0.4
        )

    ax.set_title(ttl, fontsize=17, loc="left", pad=12)
    ax.set_xlabel("t-SNE dimension 1", fontsize=12)
    ax.set_ylabel("t-SNE dimension 2", fontsize=12)
    ax.grid(True, linestyle="-", linewidth=0.4, alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Custom legend on second panel
legend_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        label=class_names[c],
        markerfacecolor=class_colors[c],
        markeredgecolor='white',
        markeredgewidth=0.8,
        markersize=10
    )
    for c in [0, 1, 2]
]

axes[1].legend(
    handles=legend_handles,
    title="Pattern class",
    loc="upper right",
    frameon=True,
    fancybox=True,
    framealpha=0.95,
    fontsize=11,
    title_fontsize=12
)

plt.tight_layout()

# -----------------------------
# 6. Save figure
# -----------------------------
png_name = "PatternNet_latent_space_comparison_tsne.png"
pdf_name = "PatternNet_latent_space_comparison_tsne.pdf"

plt.savefig(png_name, dpi=500, bbox_inches="tight")
plt.savefig(pdf_name, bbox_inches="tight")
plt.show()

print(f"Saved: {png_name}")
print(f"Saved: {pdf_name}")

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib as mpl


plt.style.use('default')
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = ['Times New Roman']
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.linewidth'] = 0.8

print("Generating Final 3D Decision Surface (HD Vector)...")

# Feature Extraction
contr_vals = Xgeo_te_s[:, 2]
trough_vals = Xgeo_te_s[:, 9]

x_min, x_max = np.percentile(contr_vals, 1), np.percentile(contr_vals, 99)
y_min, y_max = np.percentile(trough_vals, 1), np.percentile(trough_vals, 99)
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 50), np.linspace(y_min, y_max, 50))

# Simulate model logic
seq_median = np.median(Xseq_te_s, axis=0)
geo_median = np.median(Xgeo_te_s, axis=0)
grid_proba_dt = np.zeros(xx.shape)

for i in range(xx.shape[0]):
    batch_size = xx.shape[1]
    seq_batch = np.tile(seq_median, (batch_size, 1, 1)).astype(np.float32)
    geo_batch = np.tile(geo_median, (batch_size, 1)).astype(np.float32)
    geo_batch[:, 2] = xx[i, :] 
    geo_batch[:, 9] = yy[i, :] 
    preds = model.predict([seq_batch, geo_batch], verbose=0)
    grid_proba_dt[i, :] = preds[:, 2]

# Compact Figure size
fig = plt.figure(figsize=(8, 6), dpi=300) 
ax = fig.add_subplot(111, projection='3d')

# Decision Surface
surf = ax.plot_surface(xx, yy, grid_proba_dt, cmap='viridis',
                       linewidth=0.3, edgecolor='k', antialiased=True, alpha=0.6)

# Plot Points
dt_mask = (y_te == 2)
consol_mask = (y_te == 1)
noise_mask = (y_te == 0)

ax.scatter(Xgeo_te_s[dt_mask, 2], Xgeo_te_s[dt_mask, 9], proba[dt_mask, 2], 
           color='#D32F2F', s=50, label='Double Top', marker='*', edgecolor='black', linewidth=0.4, zorder=5)
ax.scatter(Xgeo_te_s[consol_mask, 2], Xgeo_te_s[consol_mask, 9], proba[consol_mask, 2], 
           color='#1976D2', s=25, label='Consolidation', marker='s', edgecolor='black', linewidth=0.4, zorder=4)
ax.scatter(Xgeo_te_s[noise_mask][:120, 2], Xgeo_te_s[noise_mask][:120, 9], proba[noise_mask][:120, 2], 
           color='#4CAF50', s=12, label='Noise', alpha=0.7, edgecolor='black', linewidth=0.3, zorder=3)

# Axis Labels - labelpad reduced to keep text inside the frame
ax.set_xlabel("Range Contraction (Scaled)", labelpad=8, fontsize=10)
ax.set_ylabel("Trough Depth (Scaled)", labelpad=8, fontsize=10)
ax.set_zlabel("Probability of Double Top", labelpad=8, fontsize=10)

# Viewing angle
ax.view_init(elev=20, azim=225)

# Legend moved slightly inward to stay safe
ax.legend(loc='upper left', bbox_to_anchor=(1.10, 0.95), frameon=True, 
          edgecolor='black', title="Classes", fontsize=10, title_fontsize=10)

# Compact Colorbar
cbar = fig.colorbar(surf, shrink=0.4, aspect=15, pad=0.05)
cbar.set_label('Confidence (DT)', rotation=270, labelpad=15, fontsize=10)

# Pane settings
ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

# Subplot adjust with generous margins
plt.subplots_adjust(left=0.15, right=0.85, bottom=0.2, top=0.9)

# Vector Graphics Save - increased pad_inches to 0.2 to catch everything
plt.savefig("3D_PatternNet_Final.pdf", format='pdf', bbox_inches='tight', pad_inches=0.2)
plt.savefig("3D_PatternNet_Final.svg", format='svg', bbox_inches='tight', pad_inches=0.2)
plt.savefig("3D_PatternNet_Final.png", format='png', dpi=600, bbox_inches='tight', pad_inches=0.2)

plt.show()
print("Saved: Clipping Fixed, Times New Roman 10pt.")

In [ ]:


import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# ---------------------------
# 1) LOSS CURVE
# ---------------------------
if "hist" not in globals():
    raise ValueError("Missing 'hist'. Run model.fit(...) first.")

H = hist.history

if "loss" in H and "val_loss" in H:
    plt.figure()
    plt.plot(H["loss"], label="Train loss")
    plt.plot(H["val_loss"], label="Validation loss")
    plt.title("Training and validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Loss keys not found in hist.history. Available keys:", list(H.keys()))

# ---------------------------
# 2) ACCURACY CURVE
# ---------------------------
# Keras sometimes stores 'accuracy' and 'val_accuracy'
acc_key = "accuracy" if "accuracy" in H else ("acc" if "acc" in H else None)
val_acc_key = "val_accuracy" if "val_accuracy" in H else ("val_acc" if "val_acc" in H else None)

if acc_key and val_acc_key:
    plt.figure()
    plt.plot(H[acc_key], label="Train accuracy")
    plt.plot(H[val_acc_key], label="Validation accuracy")
    plt.title("Training and validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Accuracy keys not found in hist.history. Available keys:", list(H.keys()))

# ---------------------------
# 3) MULTICLASS ROC (OvR)
# ---------------------------
need = ["proba", "y_te_oh", "NUM_CLASSES"]
missing = [v for v in need if v not in globals()]
if missing:
    raise ValueError(f"Missing variables: {missing}. Need proba, y_te_oh, NUM_CLASSES.")

P = np.asarray(proba)
Y = np.asarray(y_te_oh)
C = int(NUM_CLASSES)

if P.shape[0] != Y.shape[0]:
    raise ValueError("proba and y_te_oh must have same number of rows.")
if P.shape[1] != C or Y.shape[1] != C:
    raise ValueError("proba/y_te_oh second dimension must equal NUM_CLASSES.")

class_names = globals().get("class_names", ["Noise", "Consolidation", "DoubleTop"])

# Per-class ROC
fpr = {}
tpr = {}
roc_auc = {}

for i in range(C):
    fpr[i], tpr[i], _ = roc_curve(Y[:, i], P[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average ROC
fpr["micro"], tpr["micro"], _ = roc_curve(Y.ravel(), P.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Macro-average ROC
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(C)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(C):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= C
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# Plot: all ROC curves in one figure
plt.figure()
# Micro + Macro
plt.plot(fpr["micro"], tpr["micro"], label=f"micro-average (AUC = {roc_auc['micro']:.4f})")
plt.plot(fpr["macro"], tpr["macro"], label=f"macro-average (AUC = {roc_auc['macro']:.4f})")

# Per class
for i in range(C):
    nm = class_names[i] if i < len(class_names) else f"class {i}"
    plt.plot(fpr[i], tpr[i], label=f"{nm} (AUC = {roc_auc[i]:.4f})")

# Diagonal
plt.plot([0, 1], [0, 1], linestyle="--", label="chance")

plt.title("ROC curves (One-vs-Rest)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()
